# DataSprint 2.0 | CV Track
## Road damage detection with a single YOLO26m detector

| | |
|---|---|
| Task | Detect and classify road damage in street photos |
| Classes | D00 longitudinal crack, D10 transverse crack, D20 alligator crack, D40 pothole |
| Data | RDD2020 competition splits: 14,691 train, 2,150 validation, 4,200 test (normal and degraded photos) |
| Score | Accuracy (mAP50, mAP50:95, precision, recall) + speed on a T4 + robustness on degraded photos |
| Model | Ultralytics YOLO26m initialised from generic COCO weights, trained on competition data only |

**Pipeline:** data audit → label and image cleaning → preprocessing and augmentation → model selection → validation
scoring with the official scorer → speed benchmark → robustness test → test submission.

## 1. Problem Understanding
Road agencies inspect thousands of kilometres of road by hand. A detector that runs on a phone or dash camera can find
cracks and potholes automatically, so repairs are prioritised before small cracks become dangerous potholes.

* **Business goal:** accurate, fast and robust damage detection from ordinary smartphone photos.
* **Why it is hard:** thin, low-contrast cracks; very different object sizes; blurred, dark or compressed photos in the test set.
* **What good looks like:** high mAP across all four classes, under 25 ms per image on a T4, stable accuracy on degraded photos.

In [ ]:
import importlib.util, platform, subprocess, sys
if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', 'ultralytics==8.4.216'])
else:
    import ultralytics
    if ultralytics.__version__ != '8.4.216':
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', 'ultralytics==8.4.216'])
import torch, ultralytics
print('python', platform.python_version(), '| torch', torch.__version__, '| ultralytics', ultralytics.__version__,
      '| gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import os, re, json, random, hashlib, shutil
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
FIG = WORK / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
EXPECTED_SHA256 = '9f9fd2b5d51818759337daff442acdb8101ec2018be89b1deb31331d3cfde9a4'
CLASSES = ['D00 longitudinal', 'D10 transverse', 'D20 alligator', 'D40 pothole']
sns.set_theme(style='whitegrid')

REPO = next(p.parent for p in sorted(INPUT.rglob('inference.py'))
            if (p.parent / 'weights' / 'best.pt').is_file() and (p.parent / 'benchmark_speed.py').is_file())
WEIGHTS = REPO / 'weights' / 'best.pt'
sha = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()
assert sha == EXPECTED_SHA256
ROOT = next(p.parent for p in sorted(INPUT.rglob('test_image_ids.csv')) if (p.parent / 'test_images').is_dir())
print('model files :', REPO)
print('weights sha256 verified:', sha[:16], '...')
print('competition data:', ROOT)

## 2. Data Audit

In [ ]:
def load_labels(name):
    p = ROOT / name
    return pd.read_csv(p) if p.exists() else None

train_lab, val_lab = load_labels('train_labels.csv'), load_labels('val_labels.csv')
splits = {}
for split in ['train', 'val', 'test']:
    files = sorted((ROOT / f'{split}_images').glob('*'))
    splits[split] = files
labelled = {'train': train_lab, 'val': val_lab}
summary = []
for split, files in splits.items():
    lab = labelled.get(split)
    ids = {f.stem for f in files}
    row = {'split': split, 'images': len(files)}
    if lab is not None:
        row.update({'boxes': len(lab), 'images with damage': lab.image_id.nunique(),
                    'images without damage': len(ids - set(lab.image_id)),
                    'boxes per damaged image': round(len(lab) / max(lab.image_id.nunique(), 1), 2)})
    summary.append(row)
display(pd.DataFrame(summary).set_index('split'))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
cls = pd.DataFrame({s: labelled[s].class_id.value_counts().sort_index() for s in labelled if labelled[s] is not None})
cls.index = [CLASSES[i] for i in cls.index]
(cls / cls.sum() * 100).plot.bar(ax=ax[0], rot=0, color=['#4C72B0', '#DD8452'])
ax[0].set_title('Class distribution (% of boxes)')
ax[0].set_ylabel('% of boxes')
per_img = train_lab.groupby('image_id').size()
ax[1].hist(per_img.clip(upper=15), bins=range(1, 17), color='#4C72B0')
ax[1].set_title('Boxes per damaged training image')
ax[1].set_xlabel('boxes (15 = 15 or more)')
plt.tight_layout()
plt.savefig(FIG / '01_classes_and_boxes.png', dpi=150)
plt.show()
display(cls.assign(**{'train %': (cls.iloc[:, 0] / cls.iloc[:, 0].sum() * 100).round(1)}))

In [ ]:
def image_info(files):
    rows = []
    for f in files:
        try:
            with Image.open(f) as im:
                w, h = im.size
                im.verify()
            ok = True
        except Exception:
            w = h = None
            ok = False
        rows.append((f.stem, w, h, ok))
    return pd.DataFrame(rows, columns=['image_id', 'width', 'height', 'readable'])

info = {s: image_info(f) for s, f in splits.items()}
rows = []
for s, d in info.items():
    top = d.dropna().groupby(['width', 'height']).size().nlargest(3)
    rows.append({'split': s, 'images': len(d), 'unreadable': int((~d.readable).sum()),
                 'most common sizes': ', '.join(f'{int(w)}x{int(h)} ({n})' for (w, h), n in top.items())})
display(pd.DataFrame(rows).set_index('split'))

In [ ]:
def file_hash(p):
    return hashlib.md5(p.read_bytes()).hexdigest()

hashes = {s: pd.Series({f.stem: file_hash(f) for f in files}) for s, files in splits.items()}
dup_within = {s: int(h.duplicated().sum()) for s, h in hashes.items()}
cross = {'train-val': len(set(hashes['train']) & set(hashes['val'])),
         'train-test': len(set(hashes['train']) & set(hashes['test']))}
print('exact duplicate images inside each split:', dup_within)
print('identical images shared between splits:', cross)

In [ ]:
wh = info['train'].set_index('image_id')[['width', 'height']]
lab = train_lab.join(wh, on='image_id')
bw, bh = lab.x_max - lab.x_min, lab.y_max - lab.y_min
issues = pd.Series({
    'unknown class id': int((~lab.class_id.isin([0, 1, 2, 3])).sum()),
    'zero or negative width/height': int(((bw <= 0) | (bh <= 0)).sum()),
    'box outside the image': int(((lab.x_min < 0) | (lab.y_min < 0) | (lab.x_max > lab.width) | (lab.y_max > lab.height)).sum()),
    'tiny box (side < 4 px)': int(((bw < 4) | (bh < 4)).sum()),
    'exact duplicate box': int(lab.duplicated(['image_id', 'class_id', 'x_min', 'y_min', 'x_max', 'y_max']).sum()),
}, name='training boxes')
display(issues.to_frame())
rel = (bw * bh) / (lab.width * lab.height)
size_bins = pd.cut(rel, [0, 0.01, 0.05, 0.2, 1.0], labels=['tiny < 1%', 'small 1-5%', 'medium 5-20%', 'large > 20%'])
plt.figure(figsize=(8, 4))
pd.crosstab(size_bins, lab.class_id.map(dict(enumerate(CLASSES))), normalize='columns').mul(100).T.plot.barh(
    stacked=True, ax=plt.gca(), colormap='viridis')
plt.title('Box size relative to the image, per class (%)')
plt.xlabel('% of boxes')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(FIG / '02_box_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sample = train_lab.groupby('class_id').image_id.first().tolist()
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
colors = [(255, 80, 80), (80, 200, 255), (255, 200, 60), (120, 255, 120)]
for ax, iid in zip(axes, sample):
    img = cv2.cvtColor(cv2.imread(str(next((ROOT / 'train_images').glob(f'{iid}.*')))), cv2.COLOR_BGR2RGB)
    for r in train_lab[train_lab.image_id == iid].itertuples():
        cv2.rectangle(img, (int(r.x_min), int(r.y_min)), (int(r.x_max), int(r.y_max)), colors[r.class_id], 3)
        cv2.putText(img, CLASSES[r.class_id].split()[0], (int(r.x_min), max(int(r.y_min) - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, colors[r.class_id], 2)
    ax.imshow(img)
    ax.set_title(iid)
    ax.axis('off')
plt.tight_layout()
plt.savefig(FIG / '03_label_examples.png', dpi=120)
plt.show()

**Findings**
* Classes are imbalanced, so mAP is averaged per class and every class is checked separately below.
* Many boxes are small relative to the image, which is why the input resolution and the localisation quality (mAP50:95) matter.
* Part of the test set is deliberately degraded (blur, noise, compression, low light), which the robustness test covers.

## 3. Data Cleaning

In [ ]:
clean = lab.copy()
clean = clean[clean.class_id.isin([0, 1, 2, 3])]
clean['x_min'] = clean.x_min.clip(lower=0)
clean['y_min'] = clean.y_min.clip(lower=0)
clean['x_max'] = np.minimum(clean.x_max, clean.width)
clean['y_max'] = np.minimum(clean.y_max, clean.height)
clean = clean[(clean.x_max - clean.x_min >= 1) & (clean.y_max - clean.y_min >= 1)]
clean = clean.drop_duplicates(['image_id', 'class_id', 'x_min', 'y_min', 'x_max', 'y_max'])
print(f'training boxes: {len(lab):,} -> {len(clean):,} after cleaning ({len(lab) - len(clean)} removed)')

| Check | Rule | Why |
|---|---|---|
| Unreadable images | Verified every file with PIL | A corrupt file would crash training or silently drop labels |
| Duplicate images | MD5 hash of every file, within and across splits | Duplicates across splits would inflate validation scores |
| Unknown class ids | Only 0-3 kept | Guards against label noise |
| Boxes outside the image | Clipped to the image border | Keeps YOLO's normalised coordinates inside 0-1 |
| Zero-area boxes | Dropped | They carry no signal and break IoU |
| Exact duplicate boxes | Dropped | Double-counted boxes over-weight one object |
| Images without damage | **Kept** as background | They teach the model not to fire on clean road |

The Ultralytics data loader applies the same image and label verification when it builds its training cache.

## 4. Preprocessing and Augmentation

In [ ]:
ckpt = torch.load(WEIGHTS, map_location='cpu', weights_only=False)
args = ckpt.get('train_args', {}) or {}
keys = ['model', 'imgsz', 'epochs', 'batch', 'optimizer', 'lr0', 'lrf', 'cos_lr', 'warmup_epochs', 'weight_decay',
        'mosaic', 'close_mosaic', 'mixup', 'copy_paste', 'scale', 'translate', 'degrees', 'fliplr', 'flipud',
        'hsv_h', 'hsv_s', 'hsv_v', 'erasing', 'amp', 'seed']
display(pd.DataFrame({'value': {k: args.get(k) for k in keys if k in args}}))
print('best epoch:', ckpt.get('epoch'), '| saved:', ckpt.get('date'))

In [ ]:
print((REPO / 'inference.py').read_text())

| Stage | What happens | Why |
|---|---|---|
| Letterbox resize | Each photo is resized to the training size with padding, keeping its aspect ratio | No stretching of thin cracks |
| Normalisation | Pixels scaled to 0-1, RGB order | Matches the COCO pre-training |
| Mosaic (training) | 4 photos stitched into one | More objects and contexts per batch, better small-object recall |
| Scale / translate | Random zoom and shift | Damage appears at every distance from the camera |
| Horizontal flip | Mirror images | Cracks look the same mirrored; doubles variety |
| HSV colour jitter | Hue, saturation, brightness changes | Robust to sun, shade and camera differences, which helps on degraded test photos |
| Close mosaic | Mosaic switched off for the last epochs | The model finishes on natural-looking images |

Exact values are read from the checkpoint above, so they are the settings the submitted model was trained with.

## 5. Model Selection

In [ ]:
from ultralytics import YOLO
model = YOLO(str(WEIGHTS))
n_params = sum(p.numel() for p in model.model.parameters())
print(f'YOLO26m parameters: {n_params / 1e6:.1f}M | weights file: {WEIGHTS.stat().st_size / 1e6:.1f} MB | classes: {model.names}')

| Option | Verdict |
|---|---|
| Two-stage detectors (Faster R-CNN) | Accurate but far above the 25 ms speed target |
| Small one-stage models (YOLO n / s) | Fast, but miss thin, low-contrast cracks |
| **YOLO26m (chosen)** | **Latest Ultralytics generation; medium size gives strong mAP while staying under 25 ms on a T4** |
| Large models, test-time augmentation, ensembles | Small accuracy gain, but speed is 10% of the final score and they break the budget |

Initialised from generic COCO weights (allowed by the rules); trained on competition data only.

## 6. Validation with the official scorer

In [ ]:
val_pred, val_speed = WORK / 'val_pred.csv', WORK / 'val_speed.json'
subprocess.run([sys.executable, str(REPO / 'benchmark_speed.py'), '--repo', str(REPO), '--images', str(ROOT / 'val_images'),
                '--out-csv', str(val_pred), '--out-json', str(val_speed), '--warmup', '20', '--runs', '2'], check=True)
res = subprocess.run([sys.executable, str(REPO / 'score.py'), '--pred', str(val_pred), '--gt', str(ROOT / 'val_labels.csv'),
                      '--image-ids', str(ROOT / 'val_image_ids.csv')], check=True, capture_output=True, text=True)
print(res.stdout)
(WORK / 'val_score.txt').write_text(res.stdout)
speed = json.loads(val_speed.read_text())
print({k: speed[k] for k in ['gpu_name', 'median_ms', 'p95_ms', 'fps_median', 'speed_score', 'peak_gpu_mem_mb'] if k in speed})

In [ ]:
def parse(txt):
    g = lambda k: float(re.search(k + r'\s*:\s*([0-9.]+)', txt).group(1))
    per = re.findall(r'(D\d\d_\w+)\s+AP50\s+([0-9.]+)\s+AP50:95\s+([0-9.]+)', txt)
    return {'mAP50': g('mAP@50'), 'mAP50:95': g('mAP@50:95'), 'precision': g('precision'), 'recall': g('recall'),
            'accuracy score': g('accuracy score')}, pd.DataFrame(per, columns=['class', 'AP50', 'AP50:95']).astype(
            {'AP50': float, 'AP50:95': float})

clean_metrics, per_class = parse(res.stdout)
display(pd.Series(clean_metrics, name='validation').to_frame())
ax = per_class.set_index('class').plot.bar(rot=0, figsize=(10, 4.5), color=['#4C72B0', '#DD8452'])
ax.set_title('Per-class AP on validation')
ax.set_ylim(0, 1)
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f')
plt.tight_layout()
plt.savefig(FIG / '04_per_class_ap.png', dpi=150)
plt.show()

In [ ]:
preds = pd.read_csv(val_pred)
gt = val_lab
show = gt.image_id.value_counts().index[:4]
thr = float(re.search(r'confidence threshold ([0-9.]+)', res.stdout).group(1))
fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
for ax, iid in zip(axes, show):
    img = cv2.cvtColor(cv2.imread(str(next((ROOT / 'val_images').glob(f'{iid}.*')))), cv2.COLOR_BGR2RGB)
    for r in gt[gt.image_id == iid].itertuples():
        cv2.rectangle(img, (int(r.x_min), int(r.y_min)), (int(r.x_max), int(r.y_max)), (80, 255, 80), 3)
    for r in preds[(preds.image_id == iid) & (preds.score >= thr)].itertuples():
        cv2.rectangle(img, (int(r.x_min), int(r.y_min)), (int(r.x_max), int(r.y_max)), (255, 60, 60), 2)
        cv2.putText(img, f'{CLASSES[int(r.class_id)].split()[0]} {r.score:.2f}', (int(r.x_min), max(int(r.y_min) - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 60, 60), 2)
    ax.imshow(img)
    ax.set_title(f'{iid}  green = truth, red = model')
    ax.axis('off')
plt.tight_layout()
plt.savefig(FIG / '05_predictions.png', dpi=120)
plt.show()

## 7. Robustness on degraded photos

In [ ]:
rng = random.Random(0)
def degrade(img):
    h, w = img.shape[:2]
    k = rng.choice(['blur', 'noise', 'jpeg', 'dark', 'lowres'])
    if k == 'blur':
        img = cv2.GaussianBlur(img, (0, 0), rng.uniform(1.0, 2.5))
    elif k == 'noise':
        img = np.clip(img + np.random.default_rng(rng.randint(0, 10**6)).normal(0, rng.uniform(8, 20), img.shape), 0, 255).astype(np.uint8)
    elif k == 'jpeg':
        img = cv2.imdecode(cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, rng.randint(10, 25)])[1], 1)
    elif k == 'dark':
        img = np.clip(img * rng.uniform(0.35, 0.6), 0, 255).astype(np.uint8)
    else:
        s = rng.uniform(0.3, 0.5)
        img = cv2.resize(cv2.resize(img, (int(w * s), int(h * s))), (w, h))
    return img, k

deg_dir = WORK / 'val_degraded'
deg_dir.mkdir(exist_ok=True)
kinds = {}
for f in splits['val']:
    img, k = degrade(cv2.imread(str(f)))
    cv2.imwrite(str(deg_dir / f.name), img)
    kinds[f.stem] = k
deg_pred = WORK / 'val_degraded_pred.csv'
subprocess.run([sys.executable, str(REPO / 'benchmark_speed.py'), '--repo', str(REPO), '--images', str(deg_dir),
                '--out-csv', str(deg_pred), '--out-json', str(WORK / 'val_degraded_speed.json'), '--warmup', '5', '--runs', '1'],
               check=True)
res_deg = subprocess.run([sys.executable, str(REPO / 'score.py'), '--pred', str(deg_pred), '--gt', str(ROOT / 'val_labels.csv'),
                          '--image-ids', str(ROOT / 'val_image_ids.csv')], check=True, capture_output=True, text=True)
deg_metrics, _ = parse(res_deg.stdout)
robust = pd.DataFrame({'clean photos': clean_metrics, 'degraded photos': deg_metrics})
robust['kept %'] = (robust['degraded photos'] / robust['clean photos'] * 100).round(1)
display(robust)
shutil.rmtree(deg_dir)

## 8. Test predictions and submission

In [ ]:
test_pred, test_speed = WORK / 'test_pred.csv', WORK / 'test_speed.json'
subprocess.run([sys.executable, str(REPO / 'benchmark_speed.py'), '--repo', str(REPO), '--images', str(ROOT / 'test_images'),
                '--out-csv', str(test_pred), '--out-json', str(test_speed), '--warmup', '10', '--runs', '1'], check=True)
subprocess.run([sys.executable, str(REPO / 'make_kaggle_submission.py'), '--pred', str(test_pred),
                '--image-ids', str(ROOT / 'test_image_ids.csv'), '--out', str(WORK / 'submission.csv')], check=True)
subprocess.run([sys.executable, str(REPO / 'check_submission.py'), '--pred', str(WORK / 'submission.csv'),
                '--image-ids', str(ROOT / 'test_image_ids.csv')], check=True)
sub = pd.read_csv(WORK / 'submission.csv')
ts = json.loads(test_speed.read_text())
print(f"rows {len(sub)} | images with no detection {(sub.PredictionString == 'none').sum()} | "
      f"median {ts['median_ms']} ms | speed score {ts['speed_score']}")
display(sub.head())

## 9. Limitations and Future Scope
* **Thin cracks:** longitudinal and transverse cracks are thin and low-contrast, so they score lower than potholes and alligator cracks.
* **Localisation:** mAP50:95 is much lower than mAP50; crack boxes have fuzzy ends, so exact box edges are ambiguous.
* **Degraded photos:** accuracy drops on blurred, dark or compressed images (section 7).
* **Future scope:** higher input resolution or tiling for thin cracks, degradation-aware training, INT8 / TensorRT export
  for faster edge inference, and GPS tagging to build a road-health map.